# Acceptance: time-free outputs, and plain arrays from the lazy surface

**This is a sign-off page, not a tutorial.** It checks
`plans/time-free-outputs.plan.md` (composability findings `CX39` and `CX40`).

An output reduced over time — `integrate()`, `at(t)` — has no `time` dim. It used
to keep a one-point time axis, so dividing a series by its own whole-period total
raised *Output time axes differ*, and the only way round was `defer`. Now a
time-free output broadcasts across the other side's times:

| Spelling | Means |
|---|---|
| `rate / total` | share of the whole-period total at each time |
| `prev - prev.at(t)` | change since time `t` |
| `by_age / by_age.integrate().sum_by()` | each age's share of the all-age total |
| `ref.at(t)` | the value at `t`, now a lazy post-op in an `OutputSet` |
| `ref.array()`, `ref.scalar()` | the values as a plain array (the escape hatch) |
| `ref.with_values(arr)` | back from an array to an output with times, for plots and targets |

Everything below is built lazily in an `OutputSet` and evaluated after one solve.
Nothing here asserts; the checks are in `tests/test_time_free_outputs.py`. Run every
cell top to bottom (under half a minute). Each section ends with a **What to check**
box.

| | You are checking that… |
|---|---|
| A | weekly infections divided by the whole-run total give shares that add up to 1, with no `defer` |
| B | per-age shares of the all-age total keep their age labels, and children's share of infections exceeds their share of the population |
| C | `prev - prev.at(t_ref)` is zero at `t_ref` |
| D | `array()` / `scalar()` / `with_values()` give the same share as the dimensioned spelling, and `defer` over an array gives the peak |
| E | the same outputs evaluate under `jax.jit`, and the traced program does not grow with the number of save times |

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowMass,
    FlowModel,
    Output,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    Split,
    TimeAxis,
    TransitionFlow,
    defer,
)
from summer4.epi import ForceOfInfection, MixingMatrix

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult", "older"))
pmap = PropertyMap.from_property(state).stratify(age)

# Children mix mostly with children, and more than anyone else; older people least.
# Unnormalised rows, so the child row's larger total means more contacts.
contacts = np.array([[4.0, 1.0, 0.3], [1.0, 1.5, 0.4], [0.3, 0.4, 0.8]])
model = FlowModel(pmap)
foi = ForceOfInfection(
    "foi",
    infectious=state["I"],
    group_by=age,
    contact_rate=Param("beta"),
    mixing=MixingMatrix(age, contacts, normalize="none"),
)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / Param("recovery_days")))
population = {"child": 1000.0, "adult": 2500.0, "older": 800.0}
model.set_initial_population(
    {age[k]: v for k, v in population.items()},
    [Split(state, {"S": 0.999, "I": 0.001, "R": 0.0})],
)
cm = model.compile()
params = {"beta": 0.05, "recovery_days": 7.0}  # peaks near day 80; far from everyone infected
run = dict(t0=0.0, t1=181.0, dt=1.0)  # 182 daily saves: exactly 26 weeks
T_END = run["t1"]

## A. Share of the whole-run total, week by week

`daily` is new infections per day. `total` is the whole-run total: the running sum
read at the last day, `cumsum().at(T_END)`, which is time-free (dims `()`).
`weekly` is the trailing 7-day sum. Dividing either series by `total` used to raise;
now `total` broadcasts across the series' times. The bars are the weekly shares at
each week's last day (26 disjoint weeks), and the line is the running share.

The table shows each output's dims and how many times it carries: `total` has one
anchor time (the last day) and no `time` dim, and the shares keep the full daily grid.

In [ ]:
outputs = OutputSet()
outputs["daily"] = FlowMass("infection").sum_by()
outputs["total"] = outputs.ref("daily").cumsum().at(T_END)
outputs["weekly_share"] = outputs.ref("daily").rolling(7, how="sum") / outputs.ref("total")
outputs["running_share"] = outputs.ref("daily").cumsum() / outputs.ref("total")

result = cm.run(params, save=outputs.plan(), **run)
named = outputs.evaluate(result, params)

days = np.asarray(named["daily"].times.values)
week_ends = days[6::7]
weekly = pd.Series(np.asarray(named["weekly_share"].values)[6::7], index=pd.Index(week_ends, name="last day of week"), name="share of all infections")
fig = weekly.plot.bar(title="A. Share of the whole-run total, each week", labels={"value": "share of all infections"})
running = pd.Series(np.asarray(named["running_share"].values), index=days)
fig.add_scatter(x=running.index, y=running.to_numpy(), name="running share (cumsum / total)", mode="lines")
fig.show()

pd.Series(
    {
        "total: dims": named["total"].dims,
        "total: times carried": len(np.asarray(named["total"].times.values)),
        "weekly_share: dims": named["weekly_share"].dims,
        "weekly_share: times carried": len(np.asarray(named["weekly_share"].times.values)),
        "sum of the 26 weekly bars": round(float(weekly.sum()), 6),
        "running share on the last day": round(float(running.iloc[-1]), 6),
    },
    name="A",
    dtype=object,
)

> **What to check (A).** The bars rise and fall with the epidemic, and the line
> climbs from near 0 to exactly 1 on the last day. In the table, `total` has dims
> `()` and carries 1 time; `weekly_share` has dims `(time,)` and carries 182 times;
> the weekly bars sum to 1.0 and the running share ends at 1.0.

## B. Each age's share of the all-age total

`by_age` is new infections per day by age (`sum_by(age, side="dest")`), dims
`("time", "group")`. Two divisions by the all-age, whole-run total
`by_age.integrate().sum_by()` (dims `()`):

- `by_age / all_age_total` is a series per age; the stacked areas add up to the
  all-age daily share and keep the age labels;
- `by_age.integrate() / all_age_total` is time-free on both sides: each age's share
  of all infections, compared in the bar chart with each age's share of the
  population.

Here the total is `integrate()` (trapezoid), so these shares sum to 1 under that rule.

In [ ]:
outputs["by_age"] = FlowMass("infection").sum_by(age, side="dest")
outputs["all_age_total"] = outputs.ref("by_age").integrate().sum_by()
outputs["age_daily_share"] = outputs.ref("by_age") / outputs.ref("all_age_total")
outputs["attack_share"] = outputs.ref("by_age").integrate() / outputs.ref("all_age_total")

named = outputs.evaluate(cm.run(params, save=outputs.plan(), **run), params)

share = named["age_daily_share"]
labels = share.values.pmap.labels()
stacked = pd.DataFrame(np.asarray(share.values.data), index=pd.Index(days, name="day"), columns=labels)
stacked.plot.area(title="B. Each age's daily share of all infections", labels={"value": "share per day", "variable": ""}).show()

attack = named["attack_share"]
compare = pd.DataFrame(
    {
        "share of infections": np.asarray(attack.values.data),
        "share of population": np.array(list(population.values())) / sum(population.values()),
    },
    index=attack.values.pmap.labels(),
)
compare.plot.bar(barmode="group", title="B. Share of infections against share of population", labels={"index": "age", "value": "share", "variable": ""}).show()
compare.assign(**{"dims of attack_share": str(attack.dims)}).round(4)

> **What to check (B).** The area legend names the three ages (labels survived the
> division). In the bar chart the child bar for infections is clearly taller than the
> child bar for population (about 0.4 against 0.23), and the older bars the other
> way round. The
> share-of-infections column adds up to 1, and `attack_share` has dims `('group',)`.

## C. Change since a reference day

`prev - prev.at(T_REF)` subtracts the prevalence on day 50 from the whole series. `at`
is now a lazy post-op, so it works on a ref (and on a save spec).

In [ ]:
T_REF = 50.0
outputs["prev"] = Compartments(where=state["I"]).sum_by()
outputs["prev_change"] = outputs.ref("prev") - outputs.ref("prev").at(T_REF)
named = outputs.evaluate(cm.run(params, save=outputs.plan(), **run), params)

change = pd.DataFrame(
    {"prevalence": np.asarray(named["prev"].values), "prevalence - prevalence.at(50)": np.asarray(named["prev_change"].values)},
    index=pd.Index(days, name="day"),
)
fig = change.plot(title="C. Prevalence, and its change since day 50", labels={"value": "people", "variable": ""})
fig.add_vline(x=T_REF, line_dash="dot")
fig.show()
change.loc[[T_REF - 1, T_REF, T_REF + 1]]

> **What to check (C).** The change line crosses zero at the dotted line (day 50), and
> the two lines are the same shape, one shifted down. In the table the change is 0 on
> day 50.

## D. Out to plain arrays, and back

`array()` gives an output's values as a plain array (no times, dims or map);
`scalar()` does the same for a 0-d output and checks the shape. Arrays combine with
numbers, `Param`, NumPy ufuncs and `defer`. `with_values(arr)` gives an array an
output's times, dims and map again, so it can be plotted or scored by a `Target`.

`share_via_arrays` divides the daily array by the scalar total and puts the result
back on `daily`'s axes. It should equal `share_direct`, the same share written with
outputs. `peak` is `defer(jnp.max)` over the prevalence array: a named output that is
an array, so it lives in `named.arrays`.

In [ ]:
outputs["daily_integral"] = outputs.ref("daily").integrate().scalar()
outputs["share_via_arrays"] = outputs.ref("daily").with_values(
    outputs.ref("daily").array() / outputs.ref("daily_integral")
)
outputs["share_direct"] = outputs.ref("daily") / outputs.ref("daily").integrate()
outputs["peak"] = defer(jnp.max)(outputs.ref("prev").array())
outputs["log_peak_per_beta"] = np.log(outputs.ref("peak")) / Param("beta")
named = outputs.evaluate(cm.run(params, save=outputs.plan(), **run), params)

both = pd.DataFrame(
    {"share_direct (outputs)": np.asarray(named["share_direct"].values), "share_via_arrays (array → with_values)": np.asarray(named["share_via_arrays"].values)},
    index=pd.Index(days, name="day"),
)
fig = both.plot(title="D. The same share, two spellings", labels={"value": "share of the total per day", "variable": ""})
fig.data[1].update(line={"dash": "dash", "width": 3})
fig.show()

pd.Series(
    {
        "names in named.arrays": sorted(named.arrays),
        "largest |direct - via arrays|": float(np.max(np.abs(both.iloc[:, 0] - both.iloc[:, 1]))),
        "peak (defer over array)": round(float(named["peak"]), 3),
        "max of the prevalence series": round(float(np.max(np.asarray(named["prev"].values))), 3),
        "log(peak) / beta": round(float(named["log_peak_per_beta"]), 4),
    },
    name="D",
    dtype=object,
)

> **What to check (D).** The dashed line lies on the solid one. `named.arrays` lists
> `daily_integral`, `log_peak_per_beta` and `peak`; the largest difference is 0 (or
> round-off, below 1e-12); the peak equals the series maximum.

## E. Under `jit`, and the size of the traced program

The whole `OutputSet` evaluates inside `jax.jit`; the table compares a few values
with the eager run above. Then the share `rate / rate.integrate()` and the array
round trip are traced with `jax.make_jaxpr` for trajectories from 50 to 1,600 save
times. Broadcasting is decided on the host from the dims, so the number of traced
equations should not depend on the trajectory length.

In [ ]:
def solve_and_evaluate(p):
    return outputs.evaluate(cm.run(p, save=outputs.plan(), **run), p)

jitted = jax.jit(solve_and_evaluate)({k: jnp.asarray(v) for k, v in params.items()})
pd.DataFrame(
    {
        "eager": [float(np.asarray(named[n].values)[60]) for n in ("weekly_share", "share_via_arrays")] + [float(named["peak"])],
        "under jit": [float(np.asarray(jitted[n].values)[60]) for n in ("weekly_share", "share_via_arrays")] + [float(jitted["peak"])],
    },
    index=["weekly_share on day 60", "share_via_arrays on day 60", "peak"],
)

In [ ]:
def eqns(n, build):
    axis = TimeAxis(values=np.arange(n, dtype=np.float64), epoch=None, kind="explicit")
    traced = jax.make_jaxpr(lambda y: build(Output(times=axis, values=y, dims=("time",))))(jnp.ones(n))
    return len(traced.jaxpr.eqns)


lengths = [50, 100, 200, 400, 800, 1600]
sizes = pd.DataFrame(
    {
        "rate / rate.integrate()": [eqns(n, lambda r: (r / r.integrate()).values) for n in lengths],
        "rate - rate.at(t)": [eqns(n, lambda r: (r - r.at(3.0)).values) for n in lengths],
        "with_values(array() / integrate().scalar())": [eqns(n, lambda r: r.with_values(r.array() / r.integrate().scalar()).values) for n in lengths],
    },
    index=pd.Index(lengths, name="save times"),
)
sizes.plot(markers=True, log_x=True, title="E. Traced equations against trajectory length", labels={"value": "jaxpr equations", "variable": ""}).show()
sizes

> **What to check (E).** The two columns of the first table agree. In the plot every
> line is flat: the equation count is the same at 50 and at 1,600 save times.